In [ ]:
# Linear Regression Model
import pandas as pd
import numpy as np
df = pd.read_csv("/content/Bengaluru_House_Data.csv")

In [ ]:
def convert_sqft_to_num(x):
    if isinstance(x, (int, float)):
        return x
    if isinstance(x, str):
        x = x.strip()
        if ' - ' in x:
            parts = [float(s) for s in x.split(' - ')]
            return (parts[0] + parts[1]) / 2
        try:
            return float(x)
        except ValueError:
            return np.nan
    return np.nan

df['total_sqft'] = df['total_sqft'].apply(convert_sqft_to_num)
df.dropna(subset=['total_sqft'], inplace=True)
df = df.reset_index(drop=True)

In [ ]:
df['bath'] = df['bath'].fillna(df['bath'].median())

In [ ]:
features = [

'total_sqft',
'bhk',
'bath',
'location_avg_price',
'location_premium',
'sqft_per_bhk'

]

In [ ]:
target = 'price'

In [ ]:
available_features = [f for f in features if f in df.columns]
X = df[available_features].values
y = df[target].values

In [ ]:
print(X.shape)
print(y.shape)

(13274, 2)
(13274,)


In [ ]:
split = int(0.8 * len(X))
X_train = X[:split]
X_test = X[split:]
y_train = y[:split]
y_test = y[split:]

In [ ]:
sqft = 2000
location_premium = 1.2

In [ ]:
X = df[available_features].values
y = df[target].values

# Recalculate split and re-split X and y
split = int(0.8 * len(X))
X_train = X[:split]
X_test = X[split:]
y_train = y[:split]
y_test = y[split:]

X_train = X_train.astype(float)
X_test = X_test.astype(float)

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train = (X_train - mean) / std

X_test = (X_test - mean) / std

# Initialize Model
n_features = X_train.shape[1]
W = np.zeros(n_features)
b = 0

In [ ]:
# Prediction Function

# Linear Equation
def predict(X,W,b):
    return np.dot(X,W)+b

# MSE Loss Function
def mse(y_true,y_pred):
    return np.mean(
        (y_true-y_pred)**2
    )

# Gradient Descent
n = len(y_train)
# Predictions
y_pred = predict(X_train, W, b)
# Gradient of weights
dw = (-2/n) * np.dot(X_train.T,(y_train-y_pred))
# Gradient of Bias
db = (-2/n) * np.sum(y_train-y_pred)


In [ ]:
# Training Loop

# Hyperparameters
epochs = 5000
lr = 0.001

# Training
losses = []
for epoch in range(epochs):
    y_pred = predict(X_train,W,b)
    loss = mse(y_train,y_pred)
    losses.append(loss)

    n = len(y_train)

    dw = (-2/n) * np.dot(X_train.T,(y_train-y_pred))

    db = (-2/n) * np.sum(y_train-y_pred)

    W = W - lr*dw

    b = b - lr*db

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss={loss}")

# Test Predictions
test_pred = predict(X_test,W,b)

Epoch 0, Loss=32797.52448326584
Epoch 500, Loss=14632.303573709773
Epoch 1000, Loss=12671.10037131657
Epoch 1500, Loss=12433.250769721635
Epoch 2000, Loss=12401.608465983873
Epoch 2500, Loss=12397.010582024981
Epoch 3000, Loss=12396.267393632326
Epoch 3500, Loss=12396.130189133157
Epoch 4000, Loss=12396.101004035676
Epoch 4500, Loss=12396.09400710933


In [ ]:
# MAE
mae = np.mean(np.abs(y_test - test_pred))
print("MAE : ", mae)

# RMSE
rmse = np.sqrt(np.mean((y_test-test_pred)**2))
print("RMSE : ", rmse)

# R2 Score
ss_res = np.sum((y_test-test_pred)**2)
ss_tot = np.sum((y_test-y_test.mean())**2)
r2 = 1 - (ss_res /ss_tot)
print("R2 SCORE : ", r2)

MAE :  47.942691186287455
RMSE :  134.2729045843137
R2 SCORE :  0.40170074671236067


In [ ]:
# Save modal parameters
np.save("linear_weights.npy",W)
np.save("linear_bias.npy",b)
np.save("feature_mean.npy",mean)
np.save("feature_std.npy",std)

In [ ]:
# User Prediction Function
def predict_house_price(sqft, bath):
    x = np.array([sqft, bath])
    x = (x - mean) / std
    prediction = np.dot(x, W) + b
    return prediction

# Example
predicted_price = predict_house_price(1500, 2)
print(f"Predicted house price: {predicted_price}")

Predicted house price: 88.27707837472627


In [ ]:
# Generate predicted prices for all records

X_all = df[available_features].values
X_all_scaled = (X_all - mean) / std
df['predicted_price'] = (np.dot(X_all_scaled, W)+ b)

df[['price', 'predicted_price']].head()

,price,predicted_price
0,39.07,65.688077
1,120.00,235.159429
2,62.00,85.224511
3,95.00,119.651680
4,51.00,73.014240


In [ ]:
# Quoted Price
np.random.seed(42)
inflation = np.random.uniform(0.8,2.5,len(df))
df['quoted_price'] = (df['predicted_price']*inflation)

# Create Inflation Ratio
df['inflation_ratio'] = (df['quoted_price']/df['predicted_price'])

# Creating Difference Percentage
df['difference_percent'] = (abs(df['quoted_price']-df['predicted_price'])/df['predicted_price']) * 100
df[['quoted_price', 'predicted_price', 'difference_percent']].head()

# Market Gap - This line causes a KeyError because 'location_avg_price' is not in df.
# df['market_gap'] = (df['quoted_price']-df['location_avg_price'])

# Creating Area Value Index
df['area_value_index'] = (df['quoted_price']* 100000/df['total_sqft'])

# Creating Price Rank
df['price_rank'] = (df['quoted_price'].rank(pct=True)* 100)

# Creating Risk Score
df['risk_score'] = (0.4 * df['difference_percent']+25 * (df['inflation_ratio']- 1))
df['risk_score'] = np.clip(df['risk_score'],0,100)
df[['quoted_price', 'predicted_price', 'difference_percent', 'area_value_index', 'price_rank', 'risk_score']].head()

,quoted_price,predicted_price,difference_percent,area_value_index,price_rank,risk_score
0,94.375256,65.688077,43.671820,8937.050773,22.570438,28.386683
1,568.195581,235.159429,141.621432,21853.676199,97.626940,92.053931
2,174.232112,85.224511,104.438970,12099.452212,63.146000,67.885331
3,217.493183,119.651680,81.771942,14299.354580,74.401085,53.151763
4,77.777082,73.014240,6.523169,6481.423492,14.969113,4.240060


In [ ]:
# Creating Fraud Label
df['fraud'] = np.where((df['difference_percent']> 35)|(df['inflation_ratio']> 1.5),1,0)

# Verify Dataset
df[[
    'predicted_price',
    'quoted_price',
    'inflation_ratio',
    'difference_percent',
    'risk_score',
    'fraud'
]]

,predicted_price,quoted_price,inflation_ratio,difference_percent,risk_score,fraud
0,65.688077,94.375256,1.436718,43.671820,28.386683,1
1,235.159429,568.195581,2.416214,141.621432,92.053931,1
2,85.224511,174.232112,2.044390,104.438970,67.885331,1
3,119.651680,217.493183,1.817719,81.771942,53.151763,1
4,73.014240,77.777082,1.065232,6.523169,4.240060,0
...,...,...,...,...,...,...
13269,248.250564,562.501505,2.265862,126.586194,82.281026,1
13270,286.035558,310.468022,1.085418,8.541758,5.552142,0
13271,70.012548,94.908627,1.355595,35.559452,23.113644,1
13272,311.133459,340.121181,1.093168,9.316813,6.055928,0


In [ ]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,predicted_price,quoted_price,inflation_ratio,difference_percent,area_value_index,price_rank,risk_score,fraud
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056.0,2.0,1.0,39.07,65.688077,94.375256,1.436718,43.671820,8937.050773,22.570438,28.386683,1
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600.0,5.0,3.0,120.00,235.159429,568.195581,2.416214,141.621432,21853.676199,97.626940,92.053931,1
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440.0,2.0,3.0,62.00,85.224511,174.232112,2.044390,104.438970,12099.452212,63.146000,67.885331,1
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521.0,3.0,1.0,95.00,119.651680,217.493183,1.817719,81.771942,14299.354580,74.401085,53.151763,1
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200.0,2.0,1.0,51.00,73.014240,77.777082,1.065232,6.523169,6481.423492,14.969113,4.240060,0


In [ ]:
df.to_csv("fraud_detection_dataset.csv",index=False)

In [ ]:
print(df['fraud'].value_counts()) # 1 -> Fraud, 0 -> Genuine

fraud
1    8931
0    4343
Name: count, dtype: int64


In [ ]:
fraud_percentage = (
    df['fraud'].mean()
) * 100
print(f"Fraud Percentage: {fraud_percentage:.2f}%")

Fraud Percentage: 67.28%


In [ ]:
# Logistic Regression

# Defining Features
features = [
    'inflation_ratio',
    'difference_percent',
    'area_value_index',
    'price_rank',
    'risk_score'
]
target = 'fraud'

X = df[features].values
y = df[target].values

print(X.shape)
print(y.shape)

(13274, 5)
(13274,)


In [ ]:
# Train - Test - Split
split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]
y_train = y[:split]
y_test = y[split:]

# Standardization
mean_log = X_train.mean(axis=0)
std_log = X_train.std(axis=0)
X_train = (X_train - mean_log) / std_log
X_test = (X_test - mean_log) / std_log

In [ ]:
# Intializing Logistic Regression
n_features = X_train.shape[1]
W = np.zeros(n_features)
b = 0

# Sigmoid Function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Probability Function
def predict_probability(X,W,b):
    z = np.dot(X, W) + b
    return sigmoid(z)

In [ ]:
# Making the model learn using Gradient Descent

# Binary Cross Entropy Loss
def binary_loss(y_true, y_pred):
    epsilon = 1e-10
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1-y_true) * np.log(1-y_pred))

# Training Parameters
epochs = 5000
lr = 0.01
losses = []

# Training Loop
for epoch in range(epochs):
    probs = predict_probability(X_train,W,b)
    loss = binary_loss(y_train,probs)
    losses.append(loss)
    n = len(y_train)
    dw = (1/n) * np.dot(X_train.T,(probs - y_train))
    db = (1/n) * np.sum(probs - y_train)

    W = W - lr * dw
    b = b - lr * db

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss={loss:.6f}")

Epoch 0, Loss=0.693147
Epoch 500, Loss=0.259942
Epoch 1000, Loss=0.204838
Epoch 1500, Loss=0.176635
Epoch 2000, Loss=0.158597
Epoch 2500, Loss=0.145768
Epoch 3000, Loss=0.136032
Epoch 3500, Loss=0.128311
Epoch 4000, Loss=0.121989
Epoch 4500, Loss=0.116687


In [ ]:
# Prediction on Test Data
fraud_prob = predict_probability(X_test,W,b)
pred_labels = np.where(fraud_prob > 0.5,1,0)
accuracy = np.mean(pred_labels==y_test)
print(f"Accuracy : {accuracy*100:.2f}%")

# True Positive/False Positive
TP = np.sum((pred_labels == 1)&(y_test == 1))
FP = np.sum((pred_labels == 1)&(y_test == 0))
FN = np.sum((pred_labels == 0)&(y_test == 1))

# Precision
precision = TP/(TP+FP)
print(f"Precision = {precision*100:.2f}%")

# Recall
recall = TP/(TP+FN)
print(f"Recall = {recall*100:.2f}%")

# F1 Score
f1 = (2 *precision *recall)/(precision +recall)
print(f"F1 Score = {f1*100:.2f}%")

Accuracy : 97.74%
Precision = 100.00%
Recall = 96.64%
F1 Score = 98.29%


In [ ]:
np.save("logistic_weights.npy",W)
np.save("logistic_bias.npy",b)

In [ ]:
# Risk Level Generator
def risk_level(score, fraud_probability):
    if fraud_probability >= 0.8:
        return "Critical"
    elif fraud_probability >= 0.6:
        return "High"
    elif fraud_probability >= 0.3:
        return "Moderate"
    else:
        return "Low"

In [ ]:
# Example Usage
print(risk_level(20, 0.1))
print(risk_level(60, 0.5))
print(risk_level(90, 0.9))

Low
Moderate
Critical


In [ ]:
# Explainable Fraud Engine
def explain_fraud(
    inflation_ratio,
    difference_percent,
    price_rank
):
    reasons = []
    # Overpriced property
    if inflation_ratio > 1.5:
        reasons.append("Property price is more than 50% above estimated market value")
    elif inflation_ratio > 1.2:
        reasons.append("Property appears moderately overpriced")
    # Underpriced property
    elif inflation_ratio < 0.8:
        reasons.append("Property is significantly below estimated market value (possible bargain)")
    # Fair pricing
    else:
        reasons.append("Property price is close to estimated market value")
    # Large deviation
    if difference_percent > 35:
        if inflation_ratio > 1:
            reasons.append("Large positive deviation from predicted market price")
        else:
          reasons.append("Large negative deviation from predicted market price")
    # Extremely expensive property
    if price_rank > 95:
        reasons.append("Property is among the top 5% highest priced properties")
    return reasons

In [ ]:
def property_report(
    location,
    sqft,
    bhk,
    bath,
    predicted_price,
    quoted_price,
    fraud_probability,
    risk_score,
    price_rank
):
    print("\n")
    print("-----PROPERTY ANALYSIS REPORT-----")
    print()
    print(f"Location: {location}")
    print(f"Total Sqft: {sqft}")
    print(f"BHK: {bhk}")
    print(f"Bathrooms: {bath}")
    print()
    print("----------------------------------------")
    print(f"Predicted Market Price: ₹{predicted_price:.2f} Lakhs")
    print(f"Quoted Price: ₹{quoted_price:.2f} Lakhs")
    price_difference = (
        quoted_price -
        predicted_price
    )
    print(f"Price Difference: ₹{price_difference:.2f} Lakhs")
    if price_difference > 0:
        print("Quoted price is ABOVE market estimate.")
    else:
        print("Quoted price is BELOW market estimate.")
    print()
    print("----------------------------------------")
    print(f"Fraud Probability: {fraud_probability*100:.2f}%")
    print(f"Risk Score: {risk_score:.2f}")
    print(f"Risk Level: {risk_level(risk_score, fraud_probability)}")
    print()
    print("----------------------------------------")
    print("Reasons:")
    inflation_ratio = (quoted_price / predicted_price)
    difference_percent = (abs(quoted_price - predicted_price)/predicted_price) * 100
    reasons = explain_fraud(
        inflation_ratio,
        difference_percent,
        price_rank
    )
    if len(reasons) == 0:
        print("✓ No major pricing anomaly detected")
    else:
        for r in reasons:
            print("✓", r)
    print()
    print("----------------------------------------")
    lower_price = predicted_price * 0.95
    upper_price = predicted_price * 1.05
    print(f"Suggested Fair Price Range:")
    print(f"₹{lower_price:.2f} Lakhs - ₹{upper_price:.2f} Lakhs")
    print()

In [ ]:
# Dynamic Input by the User
location = input("Enter Location : ")
location_avg_price_series = df[df['location']==location]['price'].mean()
if pd.isna(location_avg_price_series):
    location_avg_price = df['price'].mean() # Fallback to global mean
else:
    location_avg_price = location_avg_price_series

location_premium = (location_avg_price/df['price'].mean())
sqft = float(input("Enter Total Sqft: "))
bhk = int(input("Enter BHK: "))
bath = int(input("Enter Bathrooms: "))
quoted_price = float(input("Enter Quoted Price (Lakhs): "))

# Predict Price
sqft_per_bhk = sqft / bhk

# Load linear regression model parameters
W_linear = np.load("linear_weights.npy")
b_linear = np.load("linear_bias.npy")

# Create feature vector for linear model prediction with only 'sqft' and 'bath' as the model was trained on these two features.
x_linear_features = np.array([sqft,bath])
x_scaled = (x_linear_features - mean) / std
predicted_price = (np.dot(x_scaled,W_linear)+ b_linear)

# Generate Fraud Features
inflation_ratio = (quoted_price/predicted_price)
difference_percent = (abs(quoted_price - predicted_price)/predicted_price) * 100
area_value_index = (quoted_price * 100000/sqft)

# Price Rank Approximation
price_rank = 50

# Risk Score
risk_score = (0.5 *difference_percent+20 *(inflation_ratio - 1))
risk_score = min(100,risk_score)

# Fraud Prediction

# Create feature vector for logistic regression
fraud_x = np.array([
    inflation_ratio,
    difference_percent,
    area_value_index,
    price_rank,
    risk_score
])
# Scale using mean_log and std_log (for logistic regression features)
fraud_x = (fraud_x - mean_log) / std_log
# Predict using logistic regression model (W and b from kernel state are for logistic regression)
fraud_probability = sigmoid(np.dot(fraud_x,W)+ b)
# Final report
property_report(
    location,
    sqft,
    bhk,
    bath,
    predicted_price,
    quoted_price,
    fraud_probability,
    risk_score,
    price_rank
)

Enter Location : electronic city
Enter Total Sqft: 1800
Enter BHK: 3
Enter Bathrooms: 2
Enter Quoted Price (Lakhs): 150


-----PROPERTY ANALYSIS REPORT-----

Location: electronic city
Total Sqft: 1800.0
BHK: 3
Bathrooms: 2

----------------------------------------
Predicted Market Price: ₹103.54 Lakhs
Quoted Price: ₹150.00 Lakhs
Price Difference: ₹46.46 Lakhs
Quoted price is ABOVE market estimate.

----------------------------------------
Fraud Probability: 66.80%
Risk Score: 31.41
Risk Level: High

----------------------------------------
Reasons:
✓ Property appears moderately overpriced
✓ Large positive deviation from predicted market price

----------------------------------------
Suggested Fair Price Range:
₹98.36 Lakhs - ₹108.72 Lakhs

